# LSTM and GRU: gates that decide what to keep

A code editor plugin reads a long line of JSON one character at a time and must say, at the end, whether a bracket is still open. You train a plain RNN, an LSTM and a GRU in PyTorch on the same lines, see the plain RNN fail, see it succeed on short lines, count the weights, and read the gates of the trained LSTM character by character.

About 10 minutes. It runs on a free Colab CPU runtime; no graphics card, account or key is needed. Training all four models takes about one to two minutes.

## 1. The characters and the lines

The next cell defines the 8 characters, the 6 possible line starts and two helpers: `make_lines` builds random lines, `encode` turns them into numbers. A line is a 4-character start, then a gap of filler characters with no brackets. The label is 1 if a bracket is still open at the end.

Look for: three example lines with gap 10, each with its label. The lesson used torch 2.15.0; your version may be different, and then your numbers may differ slightly.

In [1]:
import random, time
import torch, torch.nn as nn

print("torch", torch.__version__)

CHARS = ['{', '}', '"', 'a', '1', ':', ',', ' ']
IDX = {c: i for i, c in enumerate(CHARS)}
FILL = ['"', 'a', '1', ':', ',', ' ']
STARTS = [('{"a"', 1), ('{"1"', 1), ('{}, ', 0), ('{} "', 0), ('"a":', 0), ('"1",', 0)]
TEST_GAPS = [5, 20, 40, 80]
STEPS, BATCH, LR = 1000, 64, 0.01

def make_lines(n, gap, rng):
    lines, labels = [], []
    for _ in range(n):
        start, open_ = rng.choice(STARTS[:2]) if rng.random() < 0.5 else rng.choice(STARTS[2:])
        lines.append(start + ''.join(rng.choice(FILL) for _ in range(gap)))
        labels.append(open_)
    return lines, labels

def encode(lines):
    return torch.tensor([[IDX[c] for c in s] for s in lines])

for line, label in zip(*make_lines(3, 10, random.Random(5))):
    print(f"label {label}  ({'open' if label else 'closed'}):  {line!r}")

torch 2.15.0
label 0  (closed):  '"a":   ,":a "a'
label 1  (open):  '{"1"a:,",a" a:'
label 1  (open):  '{"1"a"a,,:aa""'


## 2. The model

This is the class from the lesson. Only `kind` changes: `getattr(nn, kind)` picks `nn.RNN`, `nn.LSTM` or `nn.GRU` by name. `out[:, -1]` keeps the output of the last character, where the plugin decides.

Look for: the weight counts of the recurrent layer. The LSTM has 4 times the plain RNN's weights (four blocks: three gates and the new value), the GRU 3 times.

In [2]:
class BracketReader(nn.Module):
    def __init__(self, kind, size=16):
        super().__init__()
        self.embed = nn.Embedding(8, 8)
        Layer = getattr(nn, kind)
        self.rnn = Layer(input_size=8,
                         hidden_size=size,
                         batch_first=True)
        self.head = nn.Linear(size, 1)

    def forward(self, chars):
        out, _ = self.rnn(self.embed(chars))
        return self.head(out[:, -1])[:, 0]

for kind in ["RNN", "LSTM", "GRU"]:
    n = sum(p.numel() for p in BracketReader(kind).rnn.parameters())
    print(f"{kind:5s} rnn_params = {n}")

RNN   rnn_params = 416
LSTM  rnn_params = 1664
GRU   rnn_params = 1248


## 3. The training loop

`train` builds 500 test lines at each test gap (5, 20, 40, 80), then trains for 1,000 steps. Each step uses 64 lines with one random gap from 0 to `max_gap`. Every 200 steps it prints the loss and the test accuracy at each gap. The data and the model are seeded, so every run of `train` with the same arguments gives the same numbers.

Look for: nothing yet. This cell only defines the functions.

In [3]:
def accuracy(model, test):
    with torch.no_grad():
        return {g: round(((model(x) > 0).float() == y).float().mean().item(), 3) for g, (x, y) in test.items()}

def train(kind, max_gap):
    rng = random.Random(0)
    test = {}
    for g in TEST_GAPS:
        xs, ys = make_lines(500, g, rng)
        test[g] = (encode(xs), torch.tensor(ys, dtype=torch.float))
    torch.manual_seed(0)
    model = BracketReader(kind)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss()
    t0 = time.time()
    for step in range(1, STEPS + 1):
        xs, ys = make_lines(BATCH, rng.randint(0, max_gap), rng)
        loss = loss_fn(model(encode(xs)), torch.tensor(ys, dtype=torch.float))
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 200 == 0:
            acc = accuracy(model, test)
            print(f'  step {step:4d}  loss {loss.item():.3f}  test_acc ' + '  '.join(f'gap{g}={a:.3f}' for g, a in acc.items()))
    n_params = sum(p.numel() for p in model.rnn.parameters())
    print(f'  done: rnn_params={n_params} seconds={time.time() - t0:.1f}')
    return model, accuracy(model, test)

results = {}

## 4. A plain RNN, trained on gaps 0 to 40

About half the test lines are open, so guessing scores about 0.5.

Look for: the loss stays near 0.69 and the test accuracy stays near 0.5 at every gap, even gap 5, for all 1,000 steps. Our run ended at 0.488, 0.518, 0.486, 0.560.

In [4]:
rnn40, results["RNN-40"] = train("RNN", 40)

  step  200  loss 0.689  test_acc gap5=0.496  gap20=0.490  gap40=0.436  gap80=0.494


  step  400  loss 0.687  test_acc gap5=0.510  gap20=0.490  gap40=0.544  gap80=0.494


  step  600  loss 0.686  test_acc gap5=0.512  gap20=0.498  gap40=0.490  gap80=0.512


  step  800  loss 0.695  test_acc gap5=0.524  gap20=0.506  gap40=0.490  gap80=0.536


  step 1000  loss 0.698  test_acc gap5=0.488  gap20=0.518  gap40=0.486  gap80=0.560
  done: rnn_params=416 seconds=1.8


## 5. An LSTM, same lines, same settings

Look for: the loss falls fast and the test accuracy is 1.000 at all four gaps by the first check at step 200, including gap 80, which is longer than any training line. The `seconds` value depends on your machine; the LSTM does four blocks of work per step, so it is slower than the plain RNN.

In [5]:
lstm40, results["LSTM-40"] = train("LSTM", 40)

  step  200  loss 0.010  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000


  step  400  loss 0.001  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000


  step  600  loss 0.001  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000


  step  800  loss 0.000  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000


  step 1000  loss 0.000  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000
  done: rnn_params=1664 seconds=3.6


## 6. A GRU, same lines, same settings

Look for: the same result as the LSTM, 1.000 at all four gaps from step 200.

In [6]:
gru40, results["GRU-40"] = train("GRU", 40)

  step  200  loss 0.001  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000


  step  400  loss 0.000  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000


  step  600  loss 0.000  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000


  step  800  loss 0.000  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000


  step 1000  loss 0.000  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000
  done: rnn_params=1248 seconds=4.1


## 7. The same plain RNN, trained only on gaps 0 to 10

Can a plain RNN store the bracket at all? Train it again with short gaps only.

Look for: 1.000 at every test gap, even gap 80. So the plain RNN can hold the bracket for 80 characters. With long gaps in training it never learned to, because the error signal faded on its way back to the bracket.

In [7]:
rnn10, results["RNN-10"] = train("RNN", 10)

  step  200  loss 0.001  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000


  step  400  loss 0.001  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000


  step  600  loss 0.000  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000


  step  800  loss 0.000  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000


  step 1000  loss 0.000  test_acc gap5=1.000  gap20=1.000  gap40=1.000  gap80=1.000
  done: rnn_params=416 seconds=1.2


## 8. All four side by side

Look for: only the plain RNN trained on gaps 0 to 40 is near 0.5. The other three rows are 1.000 everywhere.

In [8]:
print("test accuracy after 1,000 steps")
print(f"{'model':8s}" + "".join(f"  gap{g:<3d}" for g in TEST_GAPS))
for name, acc in results.items():
    print(f"{name:8s}" + "".join(f"  {acc[g]:.3f} " for g in TEST_GAPS))

test accuracy after 1,000 steps
model     gap5    gap20   gap40   gap80 
RNN-40    0.488   0.518   0.486   0.560 
LSTM-40   1.000   1.000   1.000   1.000 
GRU-40    1.000   1.000   1.000   1.000 
RNN-10    1.000   1.000   1.000   1.000 


## 9. Read the gates of the trained LSTM

`gate_trace` recomputes the trained LSTM by hand from its weights, one character at a time (PyTorch keeps the four blocks in the order input, forget, new value, output), and checks the result against `nn.LSTM`'s own output. Then the cell builds two lines with the same 40 filler characters: one starts `{"a"` (open), the other `{}, ` (closed). It picks the cell unit whose final value differs most between the two lines.

Look for: the line `unit 7` (yours may be another unit if your torch version differs).

In [9]:
def gate_trace(model, line):
    lstm = model.rnn
    W, U, b = lstm.weight_ih_l0, lstm.weight_hh_l0, lstm.bias_ih_l0 + lstm.bias_hh_l0
    H = lstm.hidden_size
    with torch.no_grad():
        x = model.embed(encode([line]))[0]
        h, c, rows = torch.zeros(H), torch.zeros(H), []
        for t in range(len(line)):
            z = W @ x[t] + U @ h + b
            i, f, g, o = torch.sigmoid(z[:H]), torch.sigmoid(z[H:2*H]), torch.tanh(z[2*H:3*H]), torch.sigmoid(z[3*H:])
            c = f * c + i * g
            h = o * torch.tanh(c)
            rows.append((line[t], i, f, o, c.clone()))
        ref, _ = lstm(model.embed(encode([line])))
        assert torch.allclose(ref[0, -1], h, atol=1e-5)
    return rows

rng = random.Random(1)
filler = ''.join(rng.choice(FILL) for _ in range(40))
opened, closed = '{"a"' + filler, '{}, ' + filler
to, tc = gate_trace(lstm40, opened), gate_trace(lstm40, closed)
unit = int((to[-1][4] - tc[-1][4]).abs().argmax())
print("by-hand recomputation matches nn.LSTM: yes")
print("unit", unit)
print("open line:  ", repr(opened))
print("closed line:", repr(closed))

by-hand recomputation matches nn.LSTM: yes
unit 7
open line:   '{"a"a,"1"::: :a":"::," :1 a,"1""" ,": a: ",a'
closed line: '{}, a,"1"::: :a":"::," :1 a,"1""" ,": a: ",a'


## 10. The trace, character by character

The cell prints the input, forget and output gates and the cell value of that unit at selected steps, then the smallest and mean forget gate over the 40 filler characters, and the model's prediction.

Look for: on the open line the forget gate stays near 1 (ours: 0.967 or more on the filler) and the cell keeps growing on one side of 0 (ours ends at -28.781). On the closed line, at step 1 (`'}'`) the forget gate drops (ours: 0.417), the cell crosses 0, and it ends on the other side (ours: +11.179). The predictions are `open` and `closed`.

In [10]:
for name, line, rows in (("open", opened, to), ("closed", closed, tc)):
    print(f"line {name}:")
    for t, (ch, i, f, o, c) in enumerate(rows):
        if t < 6 or t % 10 == 9 or t == len(rows) - 1:
            print(f'  t={t:2d} char={ch!r} input={i[unit]:.3f} forget={f[unit]:.3f} output={o[unit]:.3f} cell={c[unit]:+.3f}')
    fs = torch.stack([r[2][unit] for r in rows[4:]])
    print(f'  forget gate on the {len(fs)} filler chars: min {fs.min():.3f} mean {fs.mean():.3f}')
    with torch.no_grad():
        print(f'  prediction: {"open" if lstm40(encode([line])).item() > 0 else "closed"}')

line open:
  t= 0 char='{' input=0.789 forget=0.944 output=0.868 cell=-0.718
  t= 1 char='"' input=0.716 forget=0.974 output=0.876 cell=-1.269
  t= 2 char='a' input=0.970 forget=0.925 output=0.967 cell=-1.927
  t= 3 char='"' input=0.810 forget=0.997 output=0.976 cell=-2.715
  t= 4 char='a' input=0.987 forget=0.967 output=0.982 cell=-3.578
  t= 5 char=',' input=0.966 forget=0.984 output=0.967 cell=-4.455
  t= 9 char=':' input=0.990 forget=0.972 output=0.980 cell=-7.858
  t=19 char=':' input=0.985 forget=0.970 output=0.978 cell=-14.728
  t=29 char='1' input=0.970 forget=0.990 output=0.969 cell=-21.299
  t=39 char=':' input=0.986 forget=0.973 output=0.980 cell=-26.836
  t=43 char='a' input=0.983 forget=0.971 output=0.981 cell=-28.781
  forget gate on the 40 filler chars: min 0.967 mean 0.984
  prediction: open
line closed:
  t= 0 char='{' input=0.789 forget=0.944 output=0.868 cell=-0.718
  t= 1 char='}' input=0.956 forget=0.417 output=0.890 cell=+0.149
  t= 2 char=',' input=0.800 forget=0

## Now change it

Each cell below has one line marked `# <- edit`. Change it, run the cell again, and compare.

**Experiment 1: the forget dial by hand.** This is the toy cell from the first screen of the lesson: `c = forget * c + input * write`, with write = 1 on `{`, then `h = output * tanh(c)`. With `forget = 1` the bracket survives the 8 characters after it and h is 0.76, so the plugin says "open". Change `forget` to `0.9`, then `0.5`. The cell shrinks at every step (1 times 0.9 to the power 8 is 0.43), h falls under 0.5 and the answer turns to "closed".

In [11]:
forget = 1  # <- edit: try 0.9, then 0.5
inp, out_gate = 1, 1
line = '{"ab":12,'
c = 0.0
for ch in line:
    c = forget * c + inp * (1 if ch == "{" else 0)
h = out_gate * torch.tanh(torch.tensor(c)).item()
print(f"forget={forget}  cell at the end={c:.2f}  h={h:.2f}  plugin says: {'open' if h >= 0.5 else 'closed'}")

forget=1  cell at the end=1.00  h=0.76  plugin says: open


**Experiment 2: the size of the memory.** Change `size` from 16 to 32. Every block has input weights (size times 8), weights for the previous output (size times size) and two biases (2 times size). At 32 the plain RNN has 1,344 weights instead of 416, so the count grows faster than the size, and the LSTM and GRU stay at exactly 4 and 3 times the RNN.

In [12]:
size = 16  # <- edit: try 32, then 4
counts = {k: sum(p.numel() for p in BracketReader(k, size).rnn.parameters()) for k in ["RNN", "LSTM", "GRU"]}
for k, n in counts.items():
    print(f"size {size}  {k:5s} rnn_params = {n}  ({n / counts['RNN']:.0f} x RNN)")

size 16  RNN   rnn_params = 416  (1 x RNN)
size 16  LSTM  rnn_params = 1664  (4 x RNN)
size 16  GRU   rnn_params = 1248  (3 x RNN)


**Experiment 3: lines even longer.** The trained LSTM and the plain RNN trained on short gaps are tested on new lines with a much longer gap. Change `gap` from 80 to 300, then 1000 (the 1000 run takes a few seconds). No model saw lines that long. In our run all three stayed at 1.000 even at gap 1000. Step 10 shows why for the LSTM: the cell does not fade toward 0 on the filler, it keeps moving away from 0 on its own side, so the sign that means "open" or "closed" survives.

In [13]:
gap = 80  # <- edit: try 300, then 1000
xs, ys = make_lines(500, gap, random.Random(2))
x, y = encode(xs), torch.tensor(ys, dtype=torch.float)
with torch.no_grad():
    for name, model in (("LSTM-40", lstm40), ("GRU-40", gru40), ("RNN-10", rnn10)):
        print(f"gap {gap}  {name:8s} test accuracy = {((model(x) > 0).float() == y).float().mean().item():.3f}")

gap 80  LSTM-40  test accuracy = 1.000
gap 80  GRU-40   test accuracy = 1.000
gap 80  RNN-10   test accuracy = 1.000


## What you should see

- Step 2 prints the weight counts 416 (RNN), 1664 (LSTM) and 1248 (GRU).
- The plain RNN trained on gaps 0 to 40 stays near 0.5 test accuracy at every gap, up to step 1000.
- The LSTM and the GRU reach 1.000 at all four test gaps (5, 20, 40, 80) early in training; in our run, by step 200.
- The plain RNN trained on gaps 0 to 10 reaches 1.000, even at gap 80.
- In the gate trace, one unit's forget gate drops well below the filler values at `'}'` (ours: 0.417), and the two lines end with cell values of opposite sign.